## Étape 1 — Imports et chargement des données

Charge le portefeuille assuré depuis un fichier CSV ou génère 50 000 contrats
synthétiques calibrés sur la table TH 00-02.

### Colonnes attendues
| Colonne | Type | Description |
|---|---|---|
| `id` | str | Identifiant unique |
| `date_naissance` | date | Date de naissance |
| `date_entree` | date | Date d'entrée en observation |
| `date_sortie` | date | Date de sortie |
| `cause_sortie` | str | `'deces'` ou `'autre'` |
| `sexe` | str | `'H'` ou `'F'` |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.linalg import solve_banded
import warnings
warnings.filterwarnings('ignore')

# ── Paramètres
FILE_PATH = None
SEXE = 'H'
DATE_FIN_OBSERVATION = pd.Timestamp('2023-12-31')
LAMBDA_WH = 100

# ── Table TH/TF 00-02
_AGES_REF = np.array([20,25,30,35,40,45,50,55,60,65,70,75,80,85,90,95,100], dtype=float)
_QX_H = np.array([0.000830,0.000860,0.001100,0.001450,0.001840,0.002650,
                  0.003960,0.006180,0.009480,0.014870,0.024010,0.039840,
                  0.070050,0.120300,0.200490,0.310000,0.420000])
_QX_F = np.array([0.000340,0.000320,0.000350,0.000420,0.000560,0.000800,
                  0.001220,0.001940,0.003040,0.004860,0.007900,0.013650,
                  0.026100,0.050780,0.098400,0.170000,0.280000])

def qx_ref(age, sexe=None):
    s = sexe if sexe else SEXE
    tbl = _QX_H if s == 'H' else _QX_F
    return float(np.exp(np.interp(float(np.clip(age,20,99)), _AGES_REF, np.log(tbl))))

def qx_ref_array(ages, sexe=None):
    s = sexe if sexe else SEXE
    tbl = _QX_H if s == 'H' else _QX_F
    return np.exp(np.interp(np.clip(np.asarray(ages,float),20,99), _AGES_REF, np.log(tbl)))

if FILE_PATH is not None:
    data = pd.read_csv(FILE_PATH, parse_dates=['date_naissance','date_entree','date_sortie'])
    print('Fichier charge : ' + str(FILE_PATH))
else:
    np.random.seed(42)
    N = 50_000
    DATE_DEBUT = pd.Timestamp('2010-01-01')
    weights = np.exp(-0.5*((np.arange(35,76)-52)/12)**2)
    weights /= weights.sum()
    ages_entree = np.random.choice(np.arange(35,76), N, p=weights)
    entry_offsets = np.random.randint(0, 8*365, N)
    dates_entree = DATE_DEBUT + pd.to_timedelta(entry_offsets, unit='D')
    birth_offsets = ages_entree*365 + np.random.randint(0,365,N)
    dates_naissance = dates_entree - pd.to_timedelta(birth_offsets, unit='D')
    remaining = (DATE_FIN_OBSERVATION - dates_entree).days.astype(float)
    mu = -np.log(1 - qx_ref_array(ages_entree, SEXE))
    p_lapse = 0.08
    t_death = np.random.exponential(365.25/mu)
    t_lapse = np.random.exponential(365.25/p_lapse, N)
    is_death = t_death < t_lapse
    t_exit = np.where(is_death, t_death, t_lapse)
    censored = t_exit > remaining
    t_exit_final = np.minimum(t_exit, remaining).astype(int)
    cause_sortie = np.where(censored, 'autre', np.where(is_death, 'deces', 'autre'))
    dates_sortie = dates_entree + pd.to_timedelta(t_exit_final, unit='D')
    dates_sortie = pd.DatetimeIndex([min(d, DATE_FIN_OBSERVATION) for d in dates_sortie])
    data = pd.DataFrame({
        'id': ['C{:06d}'.format(i) for i in range(N)],
        'date_naissance': dates_naissance.normalize(),
        'date_entree': dates_entree.normalize(),
        'date_sortie': dates_sortie.normalize(),
        'cause_sortie': cause_sortie,
        'sexe': np.full(N, SEXE),
    })
    print('Donnees synthetiques generees (calibrees TH 00-02).')

print('Contrats charges   : {:,}'.format(len(data)))
print('Deces              : {:,}'.format((data['cause_sortie']=='deces').sum()))
print('Resiliations/fin   : {:,}'.format((data['cause_sortie']!='deces').sum()))
print('Sexe               : ' + SEXE)
print('Fin observation    : ' + str(DATE_FIN_OBSERVATION.date()))
print(data[['id','date_naissance','date_entree','date_sortie','cause_sortie','sexe']].head(3).to_string(index=False))
